# EDGE-FIRE 산불 데이터 인벤토리

Google Drive의 산불 이미지/라벨 데이터 폴더를 확인하는 시작 노트북입니다. 전체 파일을 메모리에 읽지 않고, 파일 목록과 일부 샘플만 점검합니다. 원본 데이터는 GitHub에 업로드하지 마세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from collections import Counter
import os, json
import pandas as pd


## 1. 데이터 폴더 지정

Drive에 `MyDrive/EDGE-FIRE/data/raw` 폴더를 만들고 데이터를 둔 뒤, 아래 경로를 실제 폴더에 맞게 바꾸세요.

In [ ]:
DATA_ROOT = Path('/content/drive/MyDrive/EDGE-FIRE/data/raw')  # 실제 폴더 경로로 수정
print('폴더 존재:', DATA_ROOT.exists())
if DATA_ROOT.exists():
    print('폴더:', DATA_ROOT)
    print('상위 항목:')
    for p in list(DATA_ROOT.iterdir())[:30]:
        kind = '폴더' if p.is_dir() else f'{p.stat().st_size / (1024**3):.2f} GB'
        print(f' - {p.name} ({kind})')
else:
    print('DATA_ROOT를 Drive의 실제 데이터 폴더 경로로 바꿔주세요.')


## 2. 파일 종류와 샘플 목록 확인

하위 폴더를 최대 5,000개 파일까지만 살펴봅니다. 대용량 데이터 전체를 열거나 복사하지 않습니다.

In [ ]:
MAX_FILES = 5000
counts = Counter()
samples = []
seen = 0

if DATA_ROOT.exists():
    for folder, _, filenames in os.walk(DATA_ROOT):
        for name in filenames:
            path = Path(folder) / name
            counts[path.suffix.lower() or '[확장자 없음]'] += 1
            if len(samples) < 20:
                samples.append(str(path))
            seen += 1
            if seen >= MAX_FILES:
                break
        if seen >= MAX_FILES:
            break
    print(f'확인한 파일 수: {seen}' + (' (최대치에 도달해 일부만 확인)' if seen >= MAX_FILES else ''))
    display(pd.DataFrame(counts.most_common(), columns=['확장자', '개수']))
    print('파일 경로 샘플:')
    for p in samples:
        print(' -', p)


## 3. 이미지와 JSON 라벨 한 쌍 확인

AI Hub 산불 데이터는 이미지와 JSON 라벨을 사용합니다. 아래 셀은 발견된 JSON 중 작은 파일 하나만 열어 최상위 구조를 확인합니다.

In [ ]:
image_exts = {'.jpg', '.jpeg', '.png'}
image_paths, json_paths = [], []
if DATA_ROOT.exists():
    for folder, _, filenames in os.walk(DATA_ROOT):
        for name in filenames:
            p = Path(folder) / name
            if p.suffix.lower() in image_exts and len(image_paths) < 5:
                image_paths.append(p)
            elif p.suffix.lower() == '.json' and len(json_paths) < 20:
                json_paths.append(p)
        if len(image_paths) >= 5 and len(json_paths) >= 20:
            break

print('찾은 이미지 샘플:', len(image_paths))
print('찾은 JSON 샘플:', len(json_paths))
print('JSON 경로 샘플:')
for p in json_paths[:10]: print(' -', p)

# JSON 하나의 크기가 50MB 이하일 때만 구조를 읽습니다.
small_json = next((p for p in json_paths if p.stat().st_size <= 50 * 1024 * 1024), None)
if small_json:
    with small_json.open(encoding='utf-8') as f:
        label = json.load(f)
    print('확인한 라벨:', small_json.name)
    print('최상위 항목:', list(label.keys()) if isinstance(label, dict) else type(label).__name__)
    if isinstance(label, dict):
        for key in ['info', 'categories', 'images', 'annotations', 'environment']:
            value = label.get(key)
            if isinstance(value, list): print(f'{key}: {len(value)}개')
            elif value is not None: print(f'{key}: {type(value).__name__}')
else:
    print('50MB 이하 JSON 샘플이 없어 구조 읽기를 건너뛰었습니다.')


## 4. 이미지 샘플 미리보기

폴더에서 찾은 첫 이미지 한 장만 표시합니다. 라벨 파일명이 이미지와 어떻게 연결되는지 데이터 가이드와 함께 확인하세요.

In [ ]:
if image_paths:
    from PIL import Image
    import matplotlib.pyplot as plt
    img = Image.open(image_paths[0]).convert('RGB')
    print('이미지:', image_paths[0].name, '| 크기:', img.size)
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print('이미지를 찾지 못했습니다. 압축을 풀었는지, DATA_ROOT가 맞는지 확인하세요.')


## 다음 단계

실행 결과에서 폴더 구조, JSON 최상위 항목, 이미지 파일명 예시를 확인한 뒤 구축활용가이드와 대조해 라벨 검증 코드를 추가하세요. `TS.z01`~`TS.zip` 같은 대용량 분할 압축 파일은 전부 필요한지 확인하기 전에는 Colab Drive에 무리하게 내려받지 마세요.